# Regression Models – Remaining 5

This notebook contains the five remaining Section C regression models:
- Decision Tree Regressor
- Random Forest Regressor
- Gradient Boosting Regressor
- Support Vector Regression (SVR)
- KNN Regressor

Use the **same preprocessed `X_train`, `X_test`, `y_train`, and `y_test`** from the main regression notebook.

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# X_train, X_test, y_train, y_test should already be created
# in your preprocessing/main regression notebook.

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

In [ ]:
# Evaluation function

def evaluate_model(name, model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)

    print(f"\n{name}")
    print("-" * 45)
    print(f"R²   : {r2:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"MAE  : {mae:.4f}")

    return {
        "Model": name,
        "R²": r2,
        "RMSE": rmse,
        "MAE": mae,
        "Predictions": y_pred,
        "Model_Object": model
    }

## 1. Decision Tree Regressor

In [ ]:
dt_model = DecisionTreeRegressor(
    random_state=42
)

dt_result = evaluate_model(
    "Decision Tree Regressor",
    dt_model
)

## 2. Random Forest Regressor

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_result = evaluate_model(
    "Random Forest Regressor",
    rf_model
)

## 3. Gradient Boosting Regressor

In [ ]:
gb_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb_result = evaluate_model(
    "Gradient Boosting Regressor",
    gb_model
)

## 4. Support Vector Regression (SVR)

In [ ]:
svr_model = SVR(
    kernel="rbf",
    C=100,
    epsilon=0.1
)

svr_result = evaluate_model(
    "Support Vector Regression (SVR)",
    svr_model
)

## 5. KNN Regressor

In [ ]:
knn_model = KNeighborsRegressor(
    n_neighbors=5,
    weights="distance",
    n_jobs=-1
)

knn_result = evaluate_model(
    "KNN Regressor",
    knn_model
)

## Comparison of the 5 Models

In [ ]:
results = [dt_result, rf_result, gb_result, svr_result, knn_result]

comparison_df = pd.DataFrame([
    {
        "Model": r["Model"],
        "R²": r["R²"],
        "RMSE": r["RMSE"],
        "MAE": r["MAE"]
    }
    for r in results
])

comparison_df = comparison_df.sort_values(
    by="R²",
    ascending=False
).reset_index(drop=True)

display(comparison_df)

## Predicted vs Actual – Best of These 5

In [ ]:
best_result = max(results, key=lambda x: x["R²"])

print("Best model among these 5:", best_result["Model"])

plt.figure(figsize=(8, 6))
plt.scatter(y_test, best_result["Predictions"], alpha=0.4)

low = min(np.min(y_test), np.min(best_result["Predictions"]))
high = max(np.max(y_test), np.max(best_result["Predictions"]))

plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Actual Delay (minutes)")
plt.ylabel("Predicted Delay (minutes)")
plt.title(f"Actual vs Predicted – {best_result['Model']}")
plt.show()

## Residual Plot – Best of These 5

In [ ]:
residuals = y_test - best_result["Predictions"]

plt.figure(figsize=(8, 6))
plt.scatter(best_result["Predictions"], residuals, alpha=0.4)
plt.axhline(0, linestyle="--")

plt.xlabel("Predicted Delay (minutes)")
plt.ylabel("Residuals")
plt.title(f"Residual Plot – {best_result['Model']}")
plt.show()

## Random Forest Feature Importance

In [ ]:
if hasattr(X_train, "columns"):
    feature_names = X_train.columns
else:
    feature_names = [f"Feature_{i+1}" for i in range(X_train.shape[1])]

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_result["Model_Object"].feature_importances_
}).sort_values("Importance", ascending=False)

display(importance_df.head(15))

plt.figure(figsize=(10, 6))
plt.barh(
    importance_df.head(15)["Feature"][::-1],
    importance_df.head(15)["Importance"][::-1]
)
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 15 Random Forest Feature Importances")
plt.show()